# 🧠 News Bias Analyzer: Custom Model Fine-Tuning Pipeline (fixed)
1. **Fake News / Factuality Classifier** – LIAR dataset (6-class truthfulness).
2. **Political Bias Classifier** – AllSides-style data (BABE fallback).
3. **ONNX dynamic INT8 quantization** for CPU inference.
4. **Packaging** into `models_bundle.zip`.

> Kaggle: **Accelerator → GPU T4**, **Internet → ON**.

In [ ]:
# 1. Install libraries (don't upgrade pip/pyarrow - it can break Kaggle's preinstalled stack)
!pip install -q "transformers>=4.46.0" "datasets>=3.0.0" evaluate accelerate scikit-learn onnx onnxruntime

In [ ]:
import os, json, shutil, glob
import torch
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: GPU is not active. Enable it in Kaggle Settings for fast training!")

OUTPUT_DIR = "./trained_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)
MAX_LEN = 256   # used for BOTH training and inference so they match

---
## 🎯 Task 1: Fake-News Classifier (LIAR)
**Fixes:** the old URLs used `/raw/main/` and assumed a `text` column; LIAR actually uses `statement`, and the
Hugging Face version stores labels in a *different integer order* than the notebook's mapping (which would have
silently scrambled the labels). Labels are now converted **by name** into one fixed ordinal scale.

In [ ]:
LIAR_ORDER = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]
liar_id2label = {i: n for i, n in enumerate(LIAR_ORDER)}
liar_label2id = {n: i for i, n in liar_id2label.items()}
LIAR_ALIASES = {"pants-on-fire": "pants-fire", "pants-on-fire!": "pants-fire", "pants fire": "pants-fire",
                "pants on fire": "pants-fire", "barely true": "barely-true", "half true": "half-true",
                "mostly true": "mostly-true"}
LIAR_TSV_COLS = ["id", "label", "statement", "subject", "speaker", "job", "state", "party",
                 "c_barely", "c_false", "c_half", "c_mostly", "c_pants", "context"]

def _load_liar_raw():
    errors = []
    # (a) Hub dataset(s) via Parquet / JSONL files (no loading scripts)
    hub_attempts = [
        lambda: load_dataset("UKPLab/liar"),
        lambda: load_dataset("parquet", data_files={
            s: f"https://huggingface.co/datasets/ucsbnlp/liar/resolve/refs%2Fconvert%2Fparquet/default/{s}/0000.parquet"
            for s in ["train", "validation", "test"]}),
    ]
    for fn in hub_attempts:
        try:
            return fn()
        except Exception as e:
            errors.append(repr(e)[:200])
    # (b) Local original LIAR TSVs (e.g. added as a Kaggle input dataset)
    for root in ["/kaggle/input", "."]:
        tr = glob.glob(f"{root}/**/train.tsv", recursive=True)
        if tr:
            d = os.path.dirname(tr[0])
            files = {"train": f"{d}/train.tsv", "validation": f"{d}/valid.tsv", "test": f"{d}/test.tsv"}
            if all(os.path.exists(p) for p in files.values()):
                return load_dataset("csv", data_files=files, delimiter="\t", column_names=LIAR_TSV_COLS)
    raise RuntimeError("Could not load LIAR. Turn Internet ON, or add a LIAR dataset "
                       "(train.tsv/valid.tsv/test.tsv) as a Kaggle input.\n" + "\n".join(errors))

def _normalize_liar_split(ds):
    cols = ds.column_names
    text_col = next((c for c in ("statement", "text", "claim") if c in cols), None)
    label_col = next((c for c in ("label", "labels", "rating") if c in cols), None)
    if text_col is None or label_col is None:
        raise KeyError(f"Unexpected LIAR columns: {cols}")
    names = getattr(ds.features[label_col], "names", None)   # ClassLabel names, if any

    def conv(ex):
        raw = ex[label_col]
        if names is not None:
            raw = names[raw]
        if isinstance(raw, str):
            key = raw.strip().lower().replace("_", "-")
            key = LIAR_ALIASES.get(key, key)
            lbl = liar_label2id[key]
        else:
            lbl = int(raw)
        return {"text": ex[text_col], "labels": lbl}

    return ds.map(conv, remove_columns=cols)

raw_liar = _load_liar_raw()
liar_dataset = DatasetDict({s: _normalize_liar_split(raw_liar[s]) for s in ["train", "validation", "test"]})
print({k: len(v) for k, v in liar_dataset.items()})
print("Train label counts:", np.bincount(liar_dataset["train"]["labels"], minlength=6))

FAKE_MODEL_NAME = "distilbert-base-uncased"
fake_tokenizer = AutoTokenizer.from_pretrained(FAKE_MODEL_NAME)

def tokenize_liar(examples):
    return fake_tokenizer(examples["text"], truncation=True, max_length=MAX_LEN)

tokenized_liar = liar_dataset.map(tokenize_liar, batched=True)
tokenized_liar.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

fake_model = AutoModelForSequenceClassification.from_pretrained(
    FAKE_MODEL_NAME, num_labels=6, id2label=liar_id2label, label2id=liar_label2id)
print("Initialized DistilBERT for Fake News.")

In [ ]:
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

fake_output_dir = os.path.join(OUTPUT_DIR, "fake_news_model")

training_args_fake = TrainingArguments(
    output_dir=fake_output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=100,
    report_to="none",
)

fake_trainer = Trainer(
    model=fake_model,
    args=training_args_fake,
    train_dataset=tokenized_liar["train"],
    eval_dataset=tokenized_liar["validation"],
    processing_class=fake_tokenizer,          # 'tokenizer=' is deprecated/removed
    data_collator=DataCollatorWithPadding(tokenizer=fake_tokenizer),
    compute_metrics=compute_metrics,
)

print("Training Fake-News classifier...")
fake_trainer.train()
print("Held-out TEST metrics:", fake_trainer.evaluate(tokenized_liar["test"], metric_key_prefix="test"))

fake_trainer.save_model(fake_output_dir)
fake_tokenizer.save_pretrained(fake_output_dir)
print(f"Saved to {fake_output_dir}")

---
## ⚖️ Task 2: Political Bias Classifier
**Fixes:** the old fallback mapped BABE's "biased" to *left* (label 0) – wrong, since BABE only says biased vs.
non-biased, not direction – and still created a 5-class head that would only ever predict 2 classes. Also, silent
defaults (`"center"`) could turn the whole dataset into one class. Now the label set adapts to the data, unknown
labels raise an error, and the class balance is printed.

In [ ]:
BIAS_ALIASES = {
    "left": 0, "far-left": 0, "far left": 0,
    "lean-left": 1, "lean left": 1, "leans left": 1, "center-left": 1, "center left": 1,
    "center": 2, "centre": 2, "neutral": 2, "mixed": 2,
    "lean-right": 3, "lean right": 3, "leans right": 3, "center-right": 3, "center right": 3,
    "right": 4, "far-right": 4, "far right": 4,
}
BIAS_5 = {0: "left", 1: "center-left", 2: "center", 3: "center-right", 4: "right"}

def load_bias_dataset():
    try:
        raw = load_dataset("liyucheng/allsides")["train"]
        cols = raw.column_names
        print("AllSides columns:", cols)
        label_col = next((c for c in ("camp", "bias", "label", "rating") if c in cols), None)
        text_cols = [c for c in ("full_stories", "articles", "text", "content", "title") if c in cols]
        if label_col is None or not text_cols:
            raise KeyError(f"Cannot find label/text columns in {cols}")
        rows, unknown = [], set()
        names = getattr(raw.features[label_col], "names", None)
        for r in raw:
            lab = r[label_col]
            if names is not None and isinstance(lab, int):
                lab = names[lab]
            key = str(lab).strip().lower()
            if key not in BIAS_ALIASES:
                unknown.add(key); continue
            text = next((r[c] for c in text_cols if isinstance(r[c], str) and len(r[c].strip()) > 30), "")
            if text:
                rows.append({"text": text[:2000], "labels": BIAS_ALIASES[key]})
        if unknown:
            print("Skipped unrecognised labels:", unknown)
        counts = np.bincount([r["labels"] for r in rows], minlength=5)
        print("Class counts:", dict(zip(BIAS_5.values(), counts)))
        if counts.max() / max(counts.sum(), 1) > 0.9 or (counts > 0).sum() < 3:
            raise ValueError("Label distribution degenerate - dataset not usable for 5-class bias.")
        return Dataset.from_list(rows), BIAS_5, "AllSides (5-class)"
    except Exception as exc:
        print(f"AllSides unavailable/unusable ({exc!r}); falling back to BABE (2-class: neutral vs biased).")
        babe = load_dataset("mediabiasgroup/BABE")["train"]
        rows = [{"text": r["text"], "labels": int(r["label"])} for r in babe
                if isinstance(r.get("text"), str) and len(r["text"].strip()) > 20 and r.get("label") in (0, 1)]
        return Dataset.from_list(rows), {0: "neutral", 1: "biased"}, "BABE (2-class)"

bias_data, bias_id2label, bias_source = load_bias_dataset()
bias_label2id = {v: k for k, v in bias_id2label.items()}
print(f"Using: {bias_source} | {len(bias_data)} samples | labels = {bias_id2label}")

# train / validation / test (validation picks the best checkpoint, test is never touched until the end)
split1 = bias_data.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)
bias_dataset = DatasetDict({"train": split1["train"], "validation": split2["train"], "test": split2["test"]})
print({k: len(v) for k, v in bias_dataset.items()})

BIAS_MODEL_NAME = "distilbert-base-uncased"
bias_tokenizer = AutoTokenizer.from_pretrained(BIAS_MODEL_NAME)

def tokenize_bias(examples):
    return bias_tokenizer(examples["text"], truncation=True, max_length=MAX_LEN)

tokenized_bias = bias_dataset.map(tokenize_bias, batched=True)
tokenized_bias.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

bias_model = AutoModelForSequenceClassification.from_pretrained(
    BIAS_MODEL_NAME, num_labels=len(bias_id2label), id2label=bias_id2label, label2id=bias_label2id)
print("Initialized DistilBERT for Political Bias.")

In [ ]:
bias_output_dir = os.path.join(OUTPUT_DIR, "bias_model")

training_args_bias = TrainingArguments(
    output_dir=bias_output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none",
)

bias_trainer = Trainer(
    model=bias_model,
    args=training_args_bias,
    train_dataset=tokenized_bias["train"],
    eval_dataset=tokenized_bias["validation"],
    processing_class=bias_tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=bias_tokenizer),
    compute_metrics=compute_metrics,
)

print("Training Political Bias classifier...")
bias_trainer.train()
print("Held-out TEST metrics:", bias_trainer.evaluate(tokenized_bias["test"], metric_key_prefix="test"))

bias_trainer.save_model(bias_output_dir)
bias_tokenizer.save_pretrained(bias_output_dir)
print(f"Saved to {bias_output_dir}")

---
## ⚡ Task 3: ONNX Dynamic INT8 Quantization
**Fixes:** newer PyTorch (≥2.5) defaults to the *dynamo* exporter, which ignores `opset_version=14`/`dynamic_axes`
behaviour and can fail on transformer models – we force the classic exporter (`dynamo=False`) and use eager
attention so tracing is reliable.

In [ ]:
import onnxruntime
from onnxruntime.quantization import quantize_dynamic, QuantType

ONNX_EXPORT_DIR = "./onnx_bundle"
os.makedirs(ONNX_EXPORT_DIR, exist_ok=True)

def export_and_quantize(model_dir, model_name):
    print(f"\nProcessing {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir, attn_implementation="eager")
    model.to("cpu").eval()

    raw_path = os.path.join(ONNX_EXPORT_DIR, f"{model_name}_raw.onnx")
    quant_path = os.path.join(ONNX_EXPORT_DIR, f"{model_name}.onnx")
    dummy = tokenizer("Sample text for ONNX tracing export.", return_tensors="pt")

    export_kwargs = dict(
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch_size", 1: "sequence_length"},
            "attention_mask": {0: "batch_size", 1: "sequence_length"},
            "logits": {0: "batch_size"},
        },
        opset_version=14,
    )
    with torch.no_grad():
        try:
            torch.onnx.export(model, (dummy["input_ids"], dummy["attention_mask"]), raw_path,
                              dynamo=False, **export_kwargs)
        except TypeError:   # older torch without the `dynamo` argument
            torch.onnx.export(model, (dummy["input_ids"], dummy["attention_mask"]), raw_path, **export_kwargs)
    print(f"Raw ONNX: {os.path.getsize(raw_path) / 1024**2:.1f} MB")

    quantize_dynamic(model_input=raw_path, model_output=quant_path, weight_type=QuantType.QInt8)
    print(f"INT8 ONNX: {os.path.getsize(quant_path) / 1024**2:.1f} MB")
    os.remove(raw_path)

    tokenizer.save_pretrained(os.path.join(ONNX_EXPORT_DIR, f"{model_name}_tokenizer"))
    with open(os.path.join(ONNX_EXPORT_DIR, f"{model_name}_labels.json"), "w") as f:
        json.dump({str(k): v for k, v in model.config.id2label.items()}, f, indent=2)

export_and_quantize(fake_output_dir, "fake_news")
export_and_quantize(bias_output_dir, "political_bias")
print("\nONNX export complete!")

---
## 🧪 Task 4: ONNX Inference Check
Also compares ONNX output to the PyTorch model to confirm quantization didn't break it.

In [ ]:
def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)   # numerically stable
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

def test_onnx_inference(text, model_name, torch_dir=None):
    model_path = os.path.join(ONNX_EXPORT_DIR, f"{model_name}.onnx")
    tok_dir = os.path.join(ONNX_EXPORT_DIR, f"{model_name}_tokenizer")
    with open(os.path.join(ONNX_EXPORT_DIR, f"{model_name}_labels.json")) as f:
        id2label = {int(k): v for k, v in json.load(f).items()}

    tokenizer = AutoTokenizer.from_pretrained(tok_dir)
    session = onnxruntime.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    inputs = tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="np")
    logits = session.run(["logits"], {
        "input_ids": inputs["input_ids"].astype(np.int64),
        "attention_mask": inputs["attention_mask"].astype(np.int64),
    })[0]
    probs = softmax(logits)
    top = int(np.argmax(probs[0]))
    result = (id2label[top], round(float(probs[0][top]), 3))

    if torch_dir:
        m = AutoModelForSequenceClassification.from_pretrained(torch_dir).eval()
        with torch.no_grad():
            pt = m(**tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="pt")).logits.numpy()
        print(f"  PyTorch top label: {id2label[int(np.argmax(pt[0]))]} (ONNX: {result[0]})")
    return result

sample = "Officials confirmed today that the proposed infrastructure legislation passed the Senate with bipartisan backing."
print("Fake-News ONNX:", test_onnx_inference(sample, "fake_news", fake_output_dir))
print("Political-Bias ONNX:", test_onnx_inference(sample, "political_bias", bias_output_dir))

---
## 📦 Task 5: Bundle

In [ ]:
shutil.make_archive("models_bundle", "zip", ONNX_EXPORT_DIR)
print(f"Created models_bundle.zip ({os.path.getsize('models_bundle.zip') / 1024**2:.1f} MB)")
print("Download it from Kaggle's Output pane and extract into backend/models/")